In [1]:
import logging

from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_ollama import ChatOllama
from langchain_ollama import OllamaEmbeddings

from langchain_chroma import Chroma

from langchain_classic.retrievers.multi_query import MultiQueryRetriever

from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

from langchain_core.prompts import ChatPromptTemplate

C:\Users\gagan\AppData\Local\Temp\ipykernel_39380\2660607351.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
c:\Users\gagan\Desktop\Langchain\labs\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
logging.basicConfig()

logging.getLogger(
    "langchain.retrievers.multi_query"
).setLevel(logging.INFO)

In [3]:
loader = WebBaseLoader(
    "https://lilianweng.github.io/posts/2023-06-23-agent/"
)

docs = loader.load()

print("Documents:", len(docs))

Documents: 1


In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

splits = text_splitter.split_documents(docs)

print("Chunks:", len(splits))

Chunks: 136


In [5]:
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

In [6]:
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory="chroma_multi_query"
)

In [7]:
base_retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 4
    }
)

In [8]:
query_llm = ChatOllama(
    model="mistral",
    temperature=0
)

In [9]:
retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=query_llm
)

In [10]:
llm = ChatOllama(
    model="mistral",
    temperature=0
)

In [11]:
prompt = ChatPromptTemplate.from_template("""
You are a helpful AI assistant.

Answer only using the provided context.

If the answer cannot be found,
say:

"I don't know based on the provided context."

Context:
{context}

Question:
{input}
""")

In [12]:
document_chain = create_stuff_documents_chain(
    llm,
    prompt
)

qa_chain = create_retrieval_chain(
    retriever,
    document_chain
)

In [13]:
question = "What is task decomposition in AI agents?"

response = qa_chain.invoke(
    {"input": question}
)

print("\nAnswer:\n")
print(response["answer"])


Answer:

 Task Decomposition in AI agents refers to the process of breaking down a large, complex task into smaller, manageable subgoals. This technique allows AI agents to efficiently handle complex tasks by focusing on one step at a time. It can be done using simple prompts, task-specific instructions, or human inputs.


In [14]:
print("\nRetrieved Documents:\n")

for i, doc in enumerate(response["context"], start=1):
    print("=" * 80)
    print(f"Document {i}")
    print("=" * 80)
    print(doc.page_content[:1000])
    print()


Retrieved Documents:

Document 1
Component One: Planning#
A complicated task usually involves many steps. An agent needs to know what they are and plan ahead.
Task Decomposition#

Document 2
Task Decomposition#
Chain of thought (CoT; Wei et al. 2022) has become a standard prompting technique for enhancing model performance on complex tasks. The model is instructed to “think step by step” to utilize more test-time computation to decompose hard tasks into smaller and simpler steps. CoT transforms big tasks into multiple manageable tasks and shed lights into an interpretation of the model’s thinking process.

Document 3
(3) Task execution: Expert models execute on the specific tasks and log results.
Instruction:

Document 4
Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs.

Document 5
Pla